In [1]:
# pip install srai

In [2]:
# pip install srai[all]

In [3]:
# pip install srai[osm]

In [4]:
# !pip install typing-extensions --upgrade

In [5]:
import pandas as pd
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf
from srai.embedders import CountEmbedder
from srai.joiners import IntersectionJoiner
from srai.loaders import OSMOnlineLoader

Wczytanie mapy Wrocławia i podzielenie obszaru na heksagony.

In [6]:
area = geocode_to_region_gdf("Wrocław, Poland")
regionalizer = H3Regionalizer(resolution=8) # https://h3geo.org/docs/core-library/restable/
regions = regionalizer.transform(area)
regions.head()

,geometry
region_id,
881e204065fffff,"POLYGON ((16.89158 51.11269, 16.88917 51.10840..."
881e204303fffff,"POLYGON ((16.86102 51.13193, 16.85861 51.12765..."
881e204763fffff,"POLYGON ((17.07169 51.13831, 17.06927 51.13403..."
881e204187fffff,"POLYGON ((16.95800 51.05916, 16.95559 51.05488..."
881e204543fffff,"POLYGON ((17.10753 51.10401, 17.10511 51.09974..."


Ściągnięcie dla Wrocławia danych o:
- sygnalizacji świetlnej,
- skrzyżowaniach,
- chodnikach,
- progach zwalniających,
- lampach ulicznych,
- drogach dla rowerów.

In [7]:
from srai.plotting import plot_regions

query = {"traffic_signals" : {"highway": "traffic_signals"},
         "crossing" : {"highway": "crossing"},
         "sidewalk" : {"footway":"sidewalk"},
         "traffic_calming" : {"traffic_calming":"yes"},
         "street_lamp" : {"highway": "street_lamp"},
         "bicycle_road" : {"bicycle_road": "yes"}
}
area = geocode_to_region_gdf("Wrocław, Poland")
loader = OSMOnlineLoader()

feature_gdf = loader.load(area, query)
feature_gdf.head()

Grouping features: 100%|██████████| 6/6 [00:00<00:00, 11.71it/s]


,geometry,traffic_signals,crossing,sidewalk,traffic_calming,street_lamp,bicycle_road
feature_id,,,,,,,
node/150597406,POINT (16.97786 51.09238),NaN,highway=crossing,NaN,NaN,NaN,NaN
node/151334674,POINT (16.97939 51.09485),highway=traffic_signals,NaN,NaN,NaN,NaN,NaN
node/153568043,POINT (16.96694 51.07828),NaN,highway=crossing,NaN,NaN,NaN,NaN
node/158719856,POINT (17.03100 51.09441),NaN,highway=crossing,NaN,NaN,NaN,NaN
node/158719876,POINT (17.04118 51.09612),highway=traffic_signals,NaN,NaN,NaN,NaN,NaN


Zliczenie ile jest w danym heksagonie sygnalizacji świetlnej, skrzyżowań itd. (nasze X).

In [8]:
regions = regionalizer.transform(area)
joiner = IntersectionJoiner()
joint_features = joiner.transform(regions, feature_gdf)
embedder = CountEmbedder()
embeddings = embedder.transform(regions, feature_gdf, joint_features)
embeddings_feature_gdf = pd.merge(regions, embeddings, on='region_id', how='inner')
embeddings_feature_gdf.head()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,geometry,traffic_signals_highway=traffic_signals,crossing_highway=crossing,sidewalk_footway=sidewalk,traffic_calming_traffic_calming=yes,street_lamp_highway=street_lamp,bicycle_road_bicycle_road=yes
region_id,,,,,,,
881e204065fffff,"POLYGON ((16.89158 51.11269, 16.88917 51.10840...",0,18,1,0,0,0
881e204303fffff,"POLYGON ((16.86102 51.13193, 16.85861 51.12765...",0,1,0,0,0,1
881e204763fffff,"POLYGON ((17.07169 51.13831, 17.06927 51.13403...",8,46,2,0,0,0
881e204187fffff,"POLYGON ((16.95800 51.05916, 16.95559 51.05488...",4,16,5,0,0,0
881e204543fffff,"POLYGON ((17.10753 51.10401, 17.10511 51.09974...",0,19,23,0,0,0


Wczytanie informacji ile wypadków było w danym heksagonie w roku 2023 (nasze Y).

In [9]:
# df_accidents = pd.read_csv('data/embeddings.csv')
df_accidents = pd.read_csv('embeddings.csv')
df_accidents.head()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<frozen importlib._bootstrap>:914: ImportWarning: APICoreClientInfoImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _OpenCVImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _AltairImportHook.find_spec() not found; falling back to find_module()


,region_id,rok_2023
0,881e204059fffff,14
1,881e205561fffff,0
2,881e2041ddfffff,3
3,881e204225fffff,10
4,881e2050a1fffff,0


Połącznie tabeli z X z tabelą z Y.

In [10]:
gdf = pd.merge(embeddings_feature_gdf, df_accidents, on='region_id', how='inner')
gdf.head()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,region_id,geometry,traffic_signals_highway=traffic_signals,crossing_highway=crossing,sidewalk_footway=sidewalk,traffic_calming_traffic_calming=yes,street_lamp_highway=street_lamp,bicycle_road_bicycle_road=yes,rok_2023
0,881e204065fffff,"POLYGON ((16.89158 51.11269, 16.88917 51.10840...",0,18,1,0,0,0,11
1,881e204303fffff,"POLYGON ((16.86102 51.13193, 16.85861 51.12765...",0,1,0,0,0,1,2
2,881e204763fffff,"POLYGON ((17.07169 51.13831, 17.06927 51.13403...",8,46,2,0,0,0,14
3,881e204187fffff,"POLYGON ((16.95800 51.05916, 16.95559 51.05488...",4,16,5,0,0,0,4
4,881e204543fffff,"POLYGON ((17.10753 51.10401, 17.10511 51.09974...",0,19,23,0,0,0,3


Dodanie kolumny mówiącej o poziomie ryzyka w danym obszarze w zależości od liczby wpadków.

In [11]:
bins = [-1, 0, 3, 10, float("inf")]  # Bins: (-1, 0], (0, 3], (3, 10], (10, inf)
label_names = ['no risk', 'low risk', 'medium risk', 'high risk']
gdf['risk_level'] = pd.cut(gdf['rok_2023'], bins=bins, labels=label_names, right=True)
gdf.head()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,region_id,geometry,traffic_signals_highway=traffic_signals,crossing_highway=crossing,sidewalk_footway=sidewalk,traffic_calming_traffic_calming=yes,street_lamp_highway=street_lamp,bicycle_road_bicycle_road=yes,rok_2023,risk_level
0,881e204065fffff,"POLYGON ((16.89158 51.11269, 16.88917 51.10840...",0,18,1,0,0,0,11,high risk
1,881e204303fffff,"POLYGON ((16.86102 51.13193, 16.85861 51.12765...",0,1,0,0,0,1,2,low risk
2,881e204763fffff,"POLYGON ((17.07169 51.13831, 17.06927 51.13403...",8,46,2,0,0,0,14,high risk
3,881e204187fffff,"POLYGON ((16.95800 51.05916, 16.95559 51.05488...",4,16,5,0,0,0,4,medium risk
4,881e204543fffff,"POLYGON ((17.10753 51.10401, 17.10511 51.09974...",0,19,23,0,0,0,3,low risk


Tworzenie grafu na podstawie powyżeszej tabeli.

In [12]:
import networkx as nx

def create_graph(weights=None):
    graph = nx.Graph()
    for index, row in gdf.iterrows():
        current_polygon = row['geometry']
        node_data = {
            'geometry': current_polygon,
            'traffic_signals_highway=traffic_signals': row['traffic_signals_highway=traffic_signals'],
            'crossing_highway=crossing': row['crossing_highway=crossing'],
            'sidewalk_footway=sidewalk': row['sidewalk_footway=sidewalk'],
            'traffic_calming_traffic_calming=yes': row['traffic_calming_traffic_calming=yes'],
            'street_lamp_highway=street_lamp': row['street_lamp_highway=street_lamp'],
            'bicycle_road_bicycle_road=yes': row['bicycle_road_bicycle_road=yes'],
            'risk_of_accident_level': row['risk_level']
        }
        graph.add_node(index, **node_data)

    for i in range(len(gdf)):
        for j in range(i + 1, len(gdf)):
            node_i = gdf.iloc[i]
            node_j = gdf.iloc[j]

            if node_i['geometry'].touches(node_j['geometry']):
                weight = 1
            else:
                try:
                    weight = nx.shortest_path_length(graph, i, j)
                except nx.NetworkXNoPath:
                    continue  # Skip this pair of nodes with no path

            graph.add_edge(i, j, weight=weight)
    return graph

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [13]:
graph = create_graph()

Kodoweanie etykiet.

In [14]:
from sklearn.preprocessing import LabelEncoder

labels = gdf['risk_level']
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

print(labels_encoded)

[0 1 0 2 1 3 0 2 3 1 3 0 2 3 1 0 0 3 3 3 0 2 1 1 2 0 3 3 0 3 0 0 3 3 1 1 1
 3 3 3 2 0 1 1 2 0 1 2 2 3 3 3 1 3 2 1 3 1 1 3 1 0 1 0 2 0 0 3 1 0 0 2 2 0
 3 3 0 2 0 3 0 1 0 0 2 0 1 2 3 1 1 0 1 3 1 3 0 0 0 2 3 1 1 0 2 3 2 0 1 3 0
 2 2 2 1 2 1 3 1 3 2 1 3 2 3 3 1 3 0 0 1 0 1 3 1 2 2 3 1 0 3 3 3 1 0 0 2 1
 3 2 0 1 3 1 2 2 3 1 1 3 1 2 3 1 2 2 0 0 3 2 3 1 3 1 2 3 0 1 3 0 2 3 3 0 3
 1 1 3 2 0 3 3 1 0 2 0 3 1 0 1 3 0 3 0 2 1 3 1 0 1 1 3 0 2 0 3 0 0 3 0 0 1
 3 2 0 0 3 0 0 3 2 3 3 0 3 1 3 3 1 1 1 3 3 2 3 2 2 2 1 1 3 3 2 1 1 0 0 2 2
 2 3 2 0 1 2 1 3 3 3 3 2 3 1 3 1 0 0 0 3 3 3 3 1 3 0 2 2 3 0 3 3 0 0 3 1 0
 1 0 2 2 0 0 3 0 0 2 3 3 0 2 3 1 0 2 1 2 3 2 0 1 0 1 1 1 2 2 3 0 3 3 1 3 3
 1 0 1 1 2 3 0 2 1 1 3 2 2 0 3 1 2 3 0 0 1 3 0 0 0 2 3 1 0 2 3 3 0 3 3 2 0
 2 3 1 0 3 0 2 3 3 2 0 3 1 1 3 3 1 0 1 0 3 0 1 2 3 2 2 3 0 0 2 0 1 3 3 0 3
 3 0 1 3 0 0 3 3 0 0 1 3 1 1 1]


In [15]:
# pip install torch_geometric

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Tworzenie grafu jako obiektu `Data` z `torch_geometric.data`.

In [16]:
import torch
from torch_geometric.data import Data
import numpy as np

feature_columns = [
    'traffic_signals_highway=traffic_signals',
    'crossing_highway=crossing',
    'sidewalk_footway=sidewalk',
    'traffic_calming_traffic_calming=yes',
    'street_lamp_highway=street_lamp',
    'bicycle_road_bicycle_road=yes'
]

node_features = gdf[feature_columns].to_numpy(dtype=np.float32)

labels = labels_encoded

edge_index = graph.edges()

edge_index = list(edge_index)
edge_index = [[x[0] for x in edge_index]+[x[1] for x in edge_index], [x[1] for x in edge_index]+[x[0] for x in edge_index]]
edge_index = torch.tensor(edge_index, dtype=torch.long)

x = torch.tensor(node_features, dtype=torch.float)
y = torch.tensor(labels, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

<frozen importlib._bootstrap>:914: ImportWarning: APICoreClientInfoImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _OpenCVImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _AltairImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: APICoreClientInfoImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _OpenCVImportHook.find_spec() not found; falling back to find_module()
<frozen imp

Statystyki grafu.

In [17]:
def print_statistics(data: Data) -> None:
    max_edges = data.num_nodes * (data.num_nodes- 1) if data.is_directed() else data.num_nodes * (data.num_nodes - 1) // 2
    edges = data.num_edges if data.is_directed() else data.num_edges // 2
    print("Statistics:")
    print("-"*50)
    print('Nodes: ', data.num_nodes)
    print('Edges: ', edges)
    print('Nodes dim: ', data.num_node_features) # wymiarowość atrybutów wierzchołków, czyli liczba cech wierzchołków
    print('Nodes class', torch.unique(data.y).size(0))
    print('Directed: ', data.is_directed())
    print('Graph density: ', round((edges / (max_edges) * 100), 3), '%')
    return

print_statistics(data=data)

Statistics:
--------------------------------------------------
Nodes:  422
Edges:  42869
Nodes dim:  6
Nodes class 4
Directed:  False
Graph density:  48.259 %


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [18]:
 #pytorch geo

---
Klasa GCNModel i SupervisedNodeClassificationGNN - kod z lab4 UR.

In [19]:
from torch import nn
from torch_geometric.nn import GCNConv

class GCNModel(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.act1 = nn.ReLU()
        self.conv2 = GCNConv(hidden_dim, out_dim)
        self.act2 = nn.ReLU()

    def forward(self, x, edge_index):
        x = self.act1(self.conv1(x, edge_index))
        x = self.act2(self.conv2(x, edge_index))
        return x

In [20]:
from typing import List, Optional, Tuple

In [21]:
import pytorch_lightning as pl
import torch
from sklearn.metrics import roc_auc_score
from torch import nn
from torch_geometric.data import Data


class SupervisedNodeClassificationGNN(pl.LightningModule):
    """Supervised node classification for a given GNN model."""

    def __init__(self, gnn: nn.Module, emb_dim: int, num_classes: int):
        super().__init__()

        self._gnn = gnn
        self._classification_head = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(emb_dim, num_classes),
            nn.LogSoftmax(dim=1),
        )

        self._loss_fn = nn.NLLLoss()

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        return self._gnn(x, edge_index)

    def training_step(self, batch: List[Data], batch_idx: int) -> torch.Tensor:
        data = batch[0]

        y_pred, y, auc = self._common_step(data=data, mask=data.train_mask)

        loss = self._loss_fn(input=y_pred, target=y)

        self.log("step", self.trainer.current_epoch)
        self.log("train/loss", loss.item(), on_epoch=True, on_step=False)
        self.log("train/auc", auc, on_epoch=True, on_step=False)

        return loss

    def validation_step(self, batch: List[Data], batch_idx: int):
        data = batch[0]

        _, _, auc = self._common_step(data=data, mask=data.val_mask)

        self.log("step", self.trainer.current_epoch)
        self.log("val/auc", auc, on_epoch=True, on_step=False)

        return {"auc": auc}

    def test_step(self, batch: List[Data], batch_idx: int):
        data = batch[0]

        _, _, auc = self._common_step(data=data, mask=data.test_mask)

        self.log("step", self.trainer.current_epoch)
        self.log("test/auc", auc, on_epoch=True, on_step=False)

        return {"auc": auc}

    def predict_step(
        self,
        batch: List[Data],
        batch_idx: int,
        dataloader_idx: Optional[int] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        data = batch[0]

        z = self(data.x, data.edge_index)
        y = data.y

        return z, y

    def _common_step(
        self,
        data: Data,
        mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, float]:
        z = self(data.x, data.edge_index)

        y_pred = self._classification_head(z)[mask]
        y = data.y[mask]

        auc = roc_auc_score(
            y_true=y,
            y_score=y_pred.exp().detach(),
            multi_class="ovr",
        )

        return y_pred, y, auc

    def configure_optimizers(self):
        return torch.optim.AdamW(
            params=self.parameters(),
            lr=1e-3,
            weight_decay=5e-4,
        )

---

In [22]:
num_features = data.x.shape[1]
hidden_dim = 256
out_dim = 128
num_classes = len(data.y.unique())

gnn = GCNModel(in_dim=num_features, hidden_dim=hidden_dim, out_dim=out_dim)
model = SupervisedNodeClassificationGNN(gnn=gnn, emb_dim=out_dim, num_classes=num_classes)


In [23]:
from torch_geometric.data import DataLoader, Data
from torch_geometric.utils import mask_select
from sklearn.model_selection import train_test_split

In [24]:
train_nodes, test_nodes = train_test_split(range(len(data.y)), test_size=0.2, random_state=42)
train_mask = []
test_mask = []
for i in range(len(data.y)):
  if i in train_nodes:
    train_mask.append(True)
  else:
    train_mask.append(False)
  if i in test_nodes:
    test_mask.append(True)
  else:
    test_mask.append(False)

In [26]:
data.test_mask = torch.tensor(test_mask)
data.val_mask = torch.tensor(test_mask)
data.train_mask = torch.tensor(train_mask)

In [27]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataListLoader

class GraphData(pl.LightningDataModule):

    def __init__(self, datamodule):
        super().__init__() # tworzy instancję klasy LightningDataModule za pomocą metody super()

        self._dataset = self._load(datamodule) # wywołuje metodę _load(dataset_name), która wczytuje i przygotowuje dane o nazwie dataset_name

    def train_dataloader(self) -> DataLoader:
        return self._dataloader()

    def val_dataloader(self) -> DataLoader:
        return self._dataloader()

    def test_dataloader(self) -> DataLoader:
        return self._dataloader()

    def predict_dataloader(self) -> DataLoader:
        return self._dataloader()

    # Metody oznaczone dekoratorem @property są używane jako właściwości i można je odczytywać bezpośrednio, bez wywoływania jako funkcje
    # na przykład, num_node_features można odczytać jako .num_node_features, metoda _load() jest statyczną metodą, co oznacza, że nie wymaga instancji klasy do jej wywołania
    # na przykład dataset = LightningDataModule._load(dataset_name="Cora")
    @property #dekorator
    def num_node_features(self) -> int: #zwraca liczbę cech węzłów (node features) dla zestawu danych _dataset
        return self._dataset.num_node_features

    @property
    def num_classes(self) -> int: # zwraca liczbę klas dla zestawu danych _dataset
        return self._dataset.num_classes

    @property
    def data(self) -> Data: # zwraca pierwszy element zestawu danych _dataset
        return self._dataset[0]

    @staticmethod
    def _load(datamodule) -> Dataset: # statyczna metoda _load(), która przyjmuje nazwę zestawu danych jako argument dataset_name
                                             # w tej metodzie sprawdzane jest, czy nazwa zestawu danych jest "Cora"
                                             # jeśli tak, używany jest Planetoid z biblioteki torch_geometric, aby wczytać i przygotować zestaw danych "Cora"
                                             # jeśli nazwa zestawu danych jest inna, zostaje zgłoszony błąd ValueError

        dataset = datamodule
        print(dataset)
        return dataset

    def _dataloader(self) -> DataLoader:
        # We can use the same DataLoader for all data splits, as there are
        # masks in the Data object that we will use for selecting the
        # appropriate nodes set. Moreover, we can set shuffle=False for all
        # splits, because we have only one Data object (there is nothing
        # to shuffle). Notice that we use PyTorch-Geometric's custom data loader
        # object, because the default PyTorch one does not know how to collate
        # Data objects in a batch.
        return DataListLoader(
            dataset=self._dataset, # określa zestaw danych, który ma być używany do tworzenia obiektu DataLoader
            batch_size=1, # określa rozmiar batcha, czyli ile przykładów danych będzie przetwarzanych równocześnie w jednym kroku
            shuffle=False, # ustawienie False oznacza, że dane nie będą mieszane przed każdą epoką treningową
                           # ponieważ jest tylko jeden obiekt Data (jeden zestaw danych), nie ma potrzeby mieszania go
            num_workers=0, # określa liczbę procesów roboczych, które mają być używane do ładowania danych w tle
                           # w tym przypadku jest ustawiony na 0, co oznacza, że ładowanie danych odbywa się w głównym wątku
        )

In [28]:
data

Data(x=[422, 6], edge_index=[2, 85738], y=[422], test_mask=[422], val_mask=[422], train_mask=[422])

In [29]:
# # Trenuj model
trainer = pl.Trainer(max_epochs=10)
datamodule_train = GraphData([data])
trainer.fit(model=model, datamodule=datamodule_train)

<frozen importlib._bootstrap>:914: ImportWarning: APICoreClientInfoImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _OpenCVImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: _AltairImportHook.find_spec() not found; falling back to find_module()
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


[Data(x=[422, 6], edge_index=[2, 85738], y=[422], test_mask=[422], val_mask=[422], train_mask=[422])]


/usr/local/lib/python3.10/dist-packages/tensorflow/python/framework/dtypes.py:35: DeprecationWarning: ml_dtypes.float8_e4m3b11 is deprecated. Use ml_dtypes.float8_e4m3b11fnuz
  from tensorflow.tsl.python.lib.core import pywrap_ml_dtypes
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                 | Type       | Params
----------------------------------------------------
0 | _gnn                 | GCNModel   | 34.7 K
1 | _classification_head | Sequential | 17.0 K
2 | _loss_fn             | NLLLoss    | 0     
----------------------------------------------------
51.7 K    Trainable params
0         Non-trainable params
51.7 K    Total params
0.207     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=1` in the `DataLoader` to improve performance.
/usr/local/lib/python3.10/dist-packages/pytorch_lightning/utilities/data.py:77: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 422. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:211: You called `self.log('step', ...)` in your `validation_step` but the value needs to be floating point. Converting it to torch.float32.
/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value o

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


---